# 02. RL 環境を触って理解する

**対応するテキスト**: [docs/04_RL環境を触って理解する.md](../docs/04_RL環境を触って理解する.md)

このノートブックは **Azure ML のコンピューティング インスタンス上（または手元の PC）で対話的に実行**します。
ジョブとしては実行しません。

目的は **学習を始める前に環境の中身を自分の目で確認する**ことです。

## 1. パッケージの導入

> ⚠ **最重要**: panda-gym は `setup.py` で **`numpy<2`** を要求しています。
> numpy 2.x が入っていると動作しません。
>
> 出典（参考情報・OSS 公式ソース）: https://github.com/qgallouedec/panda-gym/blob/master/setup.py

In [ ]:
%pip install -q "numpy<2" "gymnasium==0.29.1" "panda-gym==3.0.7" "stable-baselines3==2.4.1" imageio imageio-ffmpeg

In [ ]:
import numpy as np
import gymnasium as gym
import panda_gym   # ← これを import しないと環境 ID が登録されない（未使用に見えても必須）
import stable_baselines3 as sb3

print("numpy     :", np.__version__)
print("gymnasium :", gym.__version__)
print("panda_gym :", panda_gym.__version__)
print("sb3       :", sb3.__version__)

assert np.__version__.startswith("1."), "panda-gym は numpy<2 を要求します"
print("\nOK: numpy<2 の制約を満たしています")

## 2. Step A: `PandaReach-v3` を見る

> ⚠ **`render_mode="human"` は使わないでください。** GUI が必要なため、ヘッドレス環境では失敗します。
> 本テキストでは常に **`render_mode="rgb_array"` ＋ `renderer="Tiny"`**（PyBullet のソフトウェア描画）を使います。

In [ ]:
def make(env_id):
    """ヘッドレスで安全に環境を作る（renderer 引数が無いバージョンにも対応）。"""
    try:
        return gym.make(env_id, render_mode="rgb_array", renderer="Tiny")
    except TypeError:
        return gym.make(env_id, render_mode="rgb_array")


def describe(env_id):
    env = make(env_id)
    print(f"===== {env_id} =====")
    print("observation_space :", env.observation_space)
    print("action_space      :", env.action_space)
    print("action 次元        :", env.action_space.shape[0])
    print("max_episode_steps :", env.spec.max_episode_steps)
    obs, info = env.reset(seed=0)
    print("observation のキー :", sorted(obs.keys()))
    for k, v in obs.items():
        print(f"    {k:15s} shape={v.shape}  dtype={v.dtype}")
    print("reset 直後の info  :", info)
    env.close()
    print()


describe("PandaReach-v3")
describe("PandaPickAndPlace-v3")

### 👀 ここで必ず確認すること

| 確認項目 | 意味 |
|---|---|
| `observation_space` が **Dict** | Goal-conditioned RL の環境。**HER が使える前提条件** |
| `action_space` が **Box** | 連続値制御 → SAC / TD3 が使える |
| Reach の action 次元 = **3** | グリッパーを使わない（`block_gripper=True`） |
| PickAndPlace の action 次元 = **4** | **グリッパーの開閉が加わる** |
| `max_episode_steps` = **50** | 1 エピソードは最大 50 ステップ |

> **重要**: **目標位置は `observation` ではなく `desired_goal` に入っています。**

## 3. 【体感】疎な報酬の難しさを測る

**ランダム方策で 100 エピソード動かし、成功率を測ります。**

この差が、ベースライン実験の結果を理解する土台になります。

In [ ]:
def random_policy_stats(env_id, n_episodes=100):
    env = make(env_id)
    successes, rewards, lengths = [], [], []
    for ep in range(n_episodes):
        obs, info = env.reset(seed=1000 + ep)
        done, total, steps, ok = False, 0.0, 0, False
        while not done:
            obs, r, terminated, truncated, info = env.step(env.action_space.sample())
            total += float(r)
            steps += 1
            ok = ok or bool(info["is_success"])
            done = terminated or truncated
        successes.append(float(ok))
        rewards.append(total)
        lengths.append(steps)
    env.close()
    return {
        "env_id": env_id,
        "success_rate": float(np.mean(successes)),
        "mean_reward": float(np.mean(rewards)),
        "mean_length": float(np.mean(lengths)),
    }


for env_id in ["PandaReach-v3", "PandaPickAndPlace-v3"]:
    s = random_policy_stats(env_id)
    print(f"{s['env_id']:24s} 成功率={s['success_rate']:.3f}  "
          f"平均報酬={s['mean_reward']:7.2f}  平均長={s['mean_length']:.1f}")

### 👀 この結果の意味

- **`PandaReach-v3`**: `achieved_goal` が**手先の位置**＝エージェントが直接動かせるので、ランダムでも**たまたま成功する**
- **`PandaPickAndPlace-v3`**: `achieved_goal` が**キューブの位置**＝つかまないと動かせないので、**ほぼ成功しない**

> **重要**: PickAndPlace の成功率がほぼ 0 ということは、
> **エージェントは学習を始めるための「成功した経験」を1つも持っていない**ということです。
> これが疎な報酬の壁であり、**HER はこの壁を越えるための手法**です。
>
> **測った数値をワークシート ① に記録してください。**

## 4. 「成功」がどこで決まっているかを確認する

> **検証済みの重要な事実**（panda-gym のソースで確認）
>
> 1. `is_success` は **キューブと目標の距離が 0.05 m 未満か**だけを見る。**把持は判定しない**
> 2. 目標位置は **約 30% がテーブル上、約 70% が空中**
>
> 出典（参考情報・OSS 公式ソース）:
> https://github.com/qgallouedec/panda-gym/blob/master/panda_gym/envs/tasks/pick_and_place.py

実際に確かめます。

In [ ]:
env = make("PandaPickAndPlace-v3")
goal_z = []
for ep in range(300):
    obs, _ = env.reset(seed=2000 + ep)
    goal_z.append(float(obs["desired_goal"][2]))
env.close()

goal_z = np.array(goal_z)
on_table = goal_z < 0.03   # キューブ半分の高さ(0.02)付近 = テーブル上

print(f"目標が『テーブル上』の割合 : {on_table.mean():.3f}   （ソース上の設計値は約 0.30）")
print(f"目標が『空中』の割合       : {1 - on_table.mean():.3f}")
print(f"目標 z の範囲              : {goal_z.min():.3f} 〜 {goal_z.max():.3f}")

### ⚠ ここが改善実験の議論の核心です

> **キューブを「つかまずに押して転がす」だけでも、目標がテーブル上（約 30%）なら環境は「成功」と判定します。**
>
> つまり —
> **成功率が 30% 付近で頭打ちになっているモデルは、「押しているだけで、持ち上げを学習していない」可能性が高い。**
>
> **これはバグではなく、成功判定の設計が生む典型的な報酬ハッキングです。**
> 実務では **「報酬関数」だけでなく「成功判定そのもの」を見直す必要がある**ことを示しています。
>
> 詳しくは [docs/06_結果の読み解き.md](../docs/06_結果の読み解き.md) の 6.3 を参照してください。

## 5. HER が使える環境かを確認する

Stable-Baselines3 の HER は、環境に次の 2 つを要求します。

1. **辞書型の観測**（`observation` / `achieved_goal` / `desired_goal`）
2. **ベクトル化された `compute_reward()`**（複数ペアを配列でまとめて計算できること）

> 出典（参考情報・OSS 公式ドキュメント）: https://stable-baselines3.readthedocs.io/en/master/modules/her.html

In [ ]:
env = make("PandaPickAndPlace-v3")
obs, _ = env.reset(seed=0)

# 条件 1: 辞書型観測
assert set(obs.keys()) == {"observation", "achieved_goal", "desired_goal"}
print("条件1 OK: 辞書型観測の 3 キーが揃っています")

# 条件 2: compute_reward がバッチ入力に対応しているか
achieved = np.stack([obs["achieved_goal"]] * 5)      # (5, 3)
desired = np.stack([obs["desired_goal"]] * 5)        # (5, 3)
batched = env.compute_reward(achieved, desired, [{}] * 5)
single = env.compute_reward(obs["achieved_goal"], obs["desired_goal"], {})

print("条件2 OK: バッチ入力の戻り値 shape =", np.asarray(batched).shape)
print("          単一入力の戻り値       =", single)
print("\n→ この環境は HER が使えます。")
env.close()

## 6. ロボットの動きを「見る」

**報酬ハッキングを検出する最も確実な手段は「見ること」**です。
ただし **見る方法は実行場所によって変わります。**

| 実行場所 | 使える方法 | 指定 |
|---|---|---|
| **手元の PC / Mac** | **① GUI シミュレーター**（リアルタイム表示）／② 動画ファイル | `render_mode="human"` ／ `"rgb_array"` |
| **Azure ML Compute** | **② 動画ファイルのみ** | `render_mode="rgb_array"` ＋ `renderer="Tiny"` |

> ⚠ **クラウドでは GUI は使えません。** Azure ML のコンピューティングはヘッドレスだからです。
> 詳しくは [docs/04_RL環境を触って理解する.md](../docs/04_RL環境を触って理解する.md) の 4.6 を参照してください。

### 6-1. ① ローカルで GUI を開く（**手元の PC で実行している方のみ**）

> ⚠ **このセルは Azure ML のコンピューティング インスタンス上では動きません。**
> ヘッドレスのため OpenGL ウィンドウを開けないからです。**手元の PC で実行している場合だけ**試してください。

**GUI で確認できること／できないこと**（panda-gym のソースで確認済み）

| できること | できないこと |
|---|---|
| 3D シーンの表示 | **PyBullet のデバッグ パネル（スライダー類）** |
| カメラ操作 | **マウスでオブジェクトを掴む操作** |

panda-gym が GUI 接続の直後に `COV_ENABLE_GUI` と `COV_ENABLE_MOUSE_PICKING` を 0 にしているためです。
**「Web 記事で見たスライダーが出ない」のは不具合ではなく仕様です。**

> 出典（参考情報・OSS 公式ソース）: https://raw.githubusercontent.com/qgallouedec/panda-gym/master/panda_gym/pybullet.py

> ⚠ **`human` モードでは `render()` は `None` を返します。** 動画を残したいなら `rgb_array` を使ってください
> （`human` と `rgb_array` は同時に使えません）。

In [ ]:
# ⚠ 手元の PC で実行している場合だけ実行してください（クラウドでは失敗します）
RUN_GUI = False   # ← ローカルで試すときに True に変えてください

if RUN_GUI:
    gui_env = gym.make("PandaPickAndPlace-v3", render_mode="human")
    obs, info = gui_env.reset(seed=0)
    for _ in range(300):
        obs, r, terminated, truncated, info = gui_env.step(gui_env.action_space.sample())
        if terminated or truncated:
            obs, info = gui_env.reset()

    # human モードでは render() は None を返す（rgb_array のときだけ画像を返す仕様）
    print("human モードでの render() の戻り値:", gui_env.render())
    gui_env.close()
    print("GUI を閉じました。ランダム方策なのでキューブはほとんど動かないはずです。")
else:
    print("RUN_GUI = False のためスキップしました。")
    print("手元の PC で実行している場合は RUN_GUI = True にして、ロボットが動く様子を見てください。")

### 6-2. ② 動画ファイルを書き出す（**クラウドではこちらだけ**）

`renderer="Tiny"` は PyBullet のソフトウェア レンダラー（`p.DIRECT`）です。**GPU も X サーバーも不要**です。

> PyBullet 公式ガイドは、DIRECT モードのソフトウェア レンダラーについて
> 「**This can be useful for running simulations in the cloud on servers without GPU.**」と明記しています。
> → 出典（参考情報・OSS 公式ドキュメント）: [PyBullet Quickstart Guide](https://raw.githubusercontent.com/bulletphysics/bullet3/master/docs/pybullet_quickstart_guide/PyBulletQuickstartGuide.md.html)
>
> **これが、本ハンズオンが CPU クラスターだけで完結できる理由です。**

In [ ]:
import imageio.v2 as imageio

env = make("PandaPickAndPlace-v3")
frames = []
for ep in range(2):
    obs, _ = env.reset(seed=3000 + ep)
    done = False
    while not done:
        frame = env.render()
        assert frame is not None, "render() が None です。render_mode='rgb_array' を確認してください"
        frames.append(np.asarray(frame))
        obs, r, terminated, truncated, info = env.step(env.action_space.sample())
        done = terminated or truncated
env.close()

print("フレーム数:", len(frames), " 解像度:", frames[0].shape)

with imageio.get_writer("random_policy.mp4", fps=20) as writer:
    for f in frames:
        writer.append_data(f)
print("random_policy.mp4 を書き出しました。再生して、ランダム方策の様子を確認してください。")

## 7. ✅ チェックリスト

- [ ] `numpy` が **1.x** であることを確認した
- [ ] 両環境の `observation_space` / `action_space` / `max_episode_steps` を確認した
- [ ] Reach は action 3 次元、PickAndPlace は 4 次元であることを確認した
- [ ] **ランダム方策の成功率を両環境で実測し、ワークシート ① に記録した**
- [ ] **目標の約 30% がテーブル上**であることを実測した
- [ ] HER の 2 条件（辞書型観測・ベクトル化 `compute_reward`）を確認した
- [ ] `random_policy.mp4` を書き出して**再生した**

→ 次は [docs/05_ベースライン実験.md](../docs/05_ベースライン実験.md) と [03_baseline_job.ipynb](03_baseline_job.ipynb) へ進みます。